<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.2 MB/s eta 0:00:00


Mount The Drive

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Packages

In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Sentence Tranformer

In [4]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


Read the Data

In [5]:
chunks = []
dict = {}
#with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
#with open("/content/drive/MyDrive/AI_capstone_training_data/all_records_final_formatted_3.json", encoding="utf-8") as f:
test_rec = open("test_records.jsonl", "a")
with open("/content/drive/MyDrive/AI_capstone_training_data/positive_records_singular.json", encoding="utf-8") as f:
    records = json.load(f)
    #for record in records["All_Records"]:
    for record in records["Positive_Records"]:
      if(record["chunk_id"] not in dict):
        dict[record["chunk_id"]] = 1
        chunks.append(record)

    print(f"Distinct Records: {len(chunks)}")
json.dump(chunks,test_rec )

Distinct Records: 20691


Create Pragraphs With Metadata

In [6]:
paragraphs = [
    chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
All financial statement schedules have been omitted, since the required information is not applicable or is not present in amounts sufficient to require submission of the schedule, or because the informatio
Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
See accompanying Notes to Consolidated Financial Statements. Apple Inc. | 2025 Form 10-K | 29 Apple Inc. CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME (In millions) #

Section:8,Financial Statements and Supplementary Data
CompanyName:Apple
Ticker:AAPL
Year:2025
See accompanying Notes to Consolidated Financial Statements. Apple Inc. | 2025 Form 10-K | 30 Apple Inc. CONSOLIDATED BALANCE SHEETS (In millions, except number of shares, which are reflected in thousands, 


Create Embeddings

In [7]:


embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)
print(embeddings.shape)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

(20691, 384)


In [8]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS vectors:", index.ntotal)
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))


FAISS vectors: 20691
Number of chunks: 20691
Number of embeddings: 20691


Write Indexes

In [9]:


faiss.write_index(
    index,
    "financial_reports.index"
)

In [10]:
#Check the file

import os

size_mb = os.path.getsize(
    "financial_reports.index"
) / (1024 * 1024)

print(f"Index size: {size_mb:.2f} MB")

Index size: 30.31 MB


In [11]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")
    """tks = chunk["metadata"].split(" ")
    ticker = (tks[1]).split("-")[1]
    section = tks[len(tks)-1].split("-")[1]
    date = tks[len(tks)-2].split("-")[1]
    company = tks[2].split("-")[1]
    metadata.append({
        "ticker": ticker,
        "section": section,
        "year": date,
        "reference": chunk["reference"]
    })"""
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "reference": chunk["reference"]
    })

In [5]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

NameError: name 'metadata' is not defined

In [13]:
import os

print(
    "File size:",
    os.path.getsize("financial_reports.index"),
    "bytes"
)

File size: 31781421 bytes


Read Indexes

In [7]:
"""index = faiss.read_index(
    "/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever/financial_reports.index"
)
with open("/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever/financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)"""
#index = faiss.IndexFlatIP(384)
index = faiss.read_index(
    "/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports.index"
)
with open("/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)
#index = faiss.IndexFlatIP(384)
#index.add(embeddings)

Search For Top 50 Answers Based On Question Encoding

In [8]:
question = "What is Apple's revenue"

"""query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True#,
    #normalize_embeddings=True
)

print(query_embedding.shape)
print(query_embedding[0][:10])"""

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

scores, indices = index.search(
    query_embedding,
    50
)

print("Unique indices:", len(set(indices[0])))
print(indices[0])

Unique indices: 50
[ 350  351   32    6  359   55   54  360   53   12   10  353  368  358
  378   39  369  337    5   56  103  357   35  110  324   34  347  354
   60   65   61   51   46 3028   41  219   83  367  340   74   73  355
  362  361   84   81  365  371  352   43]


In [9]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

[[0.6725097  0.6603534  0.6573678  0.64513385 0.64182204 0.61445886
  0.6055928  0.6052366  0.6005393  0.5980278  0.5901251  0.58750176
  0.5833433  0.57899183 0.57669085 0.5723034  0.57048744 0.5675676
  0.56656104 0.5645948  0.56188196 0.55925786 0.558082   0.55777586
  0.554137   0.5535671  0.5534787  0.5523194  0.5521052  0.5517931
  0.55169666 0.55132926 0.54915917 0.54818785 0.5474608  0.54576504
  0.54546756 0.5452055  0.54331344 0.54305387 0.53878534 0.53866494
  0.5386066  0.53564113 0.5345828  0.5341631  0.5322534  0.53074896
  0.5291139  0.52872294]]
[[ 350  351   32    6  359   55   54  360   53   12   10  353  368  358
   378   39  369  337    5   56  103  357   35  110  324   34  347  354
    60   65   61   51   46 3028   41  219   83  367  340   74   73  355
   362  361   84   81  365  371  352   43]]


In [10]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    file_output = open("retrieved_results1.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    """dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]"""
    dict["reference"] = chunk["reference"]
    dict["chunk_id"] = chunk["chunk_id"]
    print(dict["reference"])
    print(dict["chunk_id"])
    json.dump(dict, file_output)





Section:7,Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
CompanyName:Apple
Ticker:AAPL
Year:2025
Apple Inc. | 2025 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions): ##TABLE_START 2025 Change 2024 Change 2023 iPhone $ 209,586 4 % $ 201,183 &#8212; % $ 200,583 Mac 33,708 12 % 29,984 2 % 29,357 iPad 28,023 5 % 26,694 (6) % 28,300 Wearables, Home and Accessories 35,686 (4) % 37,005 (7) % 39,845 Services (1)

AAPL-7-2025-17
Section:7,Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
CompanyName:Apple
Ticker:AAPL
Year:2025
Services (1) 109,158 14 % 96,169 13 % 85,200 Total net sales $ 416,161 6 % $ 391,035 2 % $ 383,285 ##TABLE_END (1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. iPhone iPhone net sales increased during 2025 compared to

In [19]:
for rank, (score, ids) in enumerate(zip(scores[0], indices[0])):
    print(
        f"{rank+1}: idx={ids}, score={score:.4f}"
    )

1: idx=350, score=0.6725
2: idx=351, score=0.6604
3: idx=32, score=0.6574
4: idx=6, score=0.6451
5: idx=359, score=0.6418
6: idx=55, score=0.6145
7: idx=54, score=0.6056
8: idx=360, score=0.6052
9: idx=53, score=0.6005
10: idx=12, score=0.5980
11: idx=10, score=0.5901
12: idx=353, score=0.5875
13: idx=368, score=0.5833
14: idx=358, score=0.5790
15: idx=378, score=0.5767
16: idx=39, score=0.5723
17: idx=369, score=0.5705
18: idx=337, score=0.5676
19: idx=5, score=0.5666
20: idx=56, score=0.5646
21: idx=103, score=0.5619
22: idx=357, score=0.5593
23: idx=35, score=0.5581
24: idx=110, score=0.5578
25: idx=324, score=0.5541
26: idx=34, score=0.5536
27: idx=347, score=0.5535
28: idx=354, score=0.5523
29: idx=60, score=0.5521
30: idx=65, score=0.5518
31: idx=61, score=0.5517
32: idx=51, score=0.5513
33: idx=46, score=0.5492
34: idx=3028, score=0.5482
35: idx=41, score=0.5475
36: idx=219, score=0.5458
37: idx=83, score=0.5455
38: idx=367, score=0.5452
39: idx=340, score=0.5433
40: idx=74, sco

Copy Results To Drive

In [16]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

All .json files copied successfully!


Evaluate FAISS

In [9]:
import numpy as np
eval_records = chunks

"""test_file = open("test_records.jsonl", "a")
for line in test_file:
         j_obj = json.loads(line)
         eval_records.append[j_obj]"""

def evaluate_faiss(
    eval_records,
    embedder,
    index,
    metadata,
    #k_values=(1, 5, 10, 20)
    k_values=(80, 100)
):
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []

    max_k = max(k_values)

    for record in eval_records:
        question = record["question"]
        correct_chunk_id = record["chunk_id"]

        # Encode query
        query_embedding = embedder.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)

        # Retrieve candidates
        scores, ids = index.search(
            query_embedding,
            max_k
        )

        retrieved_ids = [
            metadata[idx]["chunk_id"]
            for idx in ids[0]
            if idx != -1
        ]

        # Hit / Recall @ K
        for k in k_values:
            if correct_chunk_id in retrieved_ids[:k]:
                hits[k] += 1

        # Reciprocal rank
        if correct_chunk_id in retrieved_ids:
            rank = retrieved_ids.index(correct_chunk_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    n = len(eval_records)

    metrics = {}

    for k in k_values:
        metrics[f"Recall@{k}"] = hits[k] / n

    metrics["MRR"] = sum(reciprocal_ranks) / n
    print(metrics)
    return metrics

evaluate_faiss(eval_records,
    embedder,
    index,
    metadata,

)

{'Recall@80': 0.8624039437436567, 'Recall@100': 0.8773863032236238, 'MRR': 0.44355264028802516}


{'Recall@80': 0.8624039437436567,
 'Recall@100': 0.8773863032236238,
 'MRR': 0.44355264028802516}

Evaluate FAISS And Re-Ranker